In [5]:
import random
import requests
from datetime import datetime
import time
import os
from dotenv import load_dotenv
import pandas as pd
import logging
from pathlib import Path
from tqdm import tqdm

load_dotenv()
log_path = Path("/app/logs/error.log")
log_path.parent.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.ERROR,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_path, encoding="utf-8"),
        logging.StreamHandler()
    ],
    force=True,
)

ACCOUNT = os.getenv('STOCK_ACCOUNT_USERNAME')
PASSWORD = os.getenv('STOCK_ACCOUNT_PASSWORD')
PG_URI = os.getenv('POSTGRES_URI')

In [6]:
import sys
from pathlib import Path

workspace_root = Path("/app")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from trade.stock_api import get_taiwan_stock_data, get_twse_stock_data

## 01 Stock overview

In [7]:
stock_info = pd.read_json('http://140.116.86.242:8081/api/stock/get_stock_list')
stock_info.head()

,result,data
0,success,"{'code': '0050', 'name': '元大台灣50', 'listing_ti..."
1,success,"{'code': '0051', 'name': '元大中型100', 'listing_t..."
2,success,"{'code': '0052', 'name': '富邦科技', 'listing_time..."
3,success,"{'code': '0053', 'name': '元大電子', 'listing_time..."
4,success,"{'code': '0054', 'name': '元大台商50', 'listing_ti..."


In [8]:
import requests
from requests.exceptions import JSONDecodeError

def safe_get_json(url: str, timeout: int = 15):
    """Fetch URL and parse JSON safely with clear diagnostics."""
    try:
        resp = requests.get(url, timeout=timeout)
    except requests.RequestException as e:
        print(f"Request failed: {e}")
        return None

    print(f"HTTP {resp.status_code} | Content-Type: {resp.headers.get('Content-Type', 'N/A')}")

    if not resp.text or not resp.text.strip():
        print("Empty response body (this is the common cause of JSONDecodeError at char 0).")
        return None

    try:
        return resp.json()
    except JSONDecodeError:
        print("Response is not valid JSON. First 500 chars:")
        print(resp.text[:500])
        return None

stock_list_url = 'http://140.116.86.242:8081/api/stock/get_stock_list'
stock_list_json = safe_get_json(stock_list_url)

if stock_list_json is not None:
    stock_info = pd.DataFrame(stock_list_json)
    stock_info.head()

HTTP 200 | Content-Type: application/json; charset=utf-8


In [9]:
from FinMind.data import DataLoader

api = DataLoader()
# api.login_by_token(api_token='token')
df = api.taiwan_stock_info_with_warrant()
df.head()

2026-04-21 06:43:42.511 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-04-21 06:43:42.512 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockInfoWithWarrant, data_id: 


,industry_category,stock_id,stock_name,type,date
0,封閉式基金,0001,鴻運,twse,2024-12-05
1,封閉式基金,0002,福元,twse,2024-12-05
2,封閉式基金,0003,成長,twse,2024-12-05
3,封閉式基金,0004,國民,twse,2024-12-05
4,封閉式基金,0005,成功,twse,2024-12-05


In [10]:
print(df['industry_category'].unique())

['封閉式基金' '全部(不含大盤、指數、權證、牛熊證、可展延牛熊證)' '全部(不含大盤、指數)' 'ETF' '上櫃ETF'
 '上櫃指數股票型基金(ETF)' '受益證券' 'ETN' '指數投資證券(ETN)' '認購權證(不含牛證)' '熊證(不含可展延熊證)'
 '牛證(不含可展延牛證)' '認售權證(不含熊證)' '可展延牛證' '水泥工業' '其他' '食品工業' '電器電纜' '農業科技業'
 '觀光事業' '觀光餐旅' '塑膠工業' '建材營造' '汽車工業' '電子零組件類' '紡織纖維' '貿易百貨' '運動休閒' '電子工業'
 '電子零組件業' '電機機械' '可轉換公司債' '生技醫療類' '電腦及週邊類' '運動休閒類' '化學生技醫療' '生技醫療業' '化學工業'
 '其他電子類' '玻璃陶瓷' '造紙工業' '鋼鐵工業' '居家生活' '綠能環保' '橡膠工業' '航運業' '創新板股票' '創新版股票'
 '電腦及週邊設備業' '半導體業' '其他電子業' '通信網路業' '光電業' '電子通路業' '資訊服務業' '油電燃氣業' '數位雲端類'
 '金融保險' '居家生活類' '文化創意業' '光電業類' '半導體類' '綠能環保類' '通信網路類' '電子商務業' '數位雲端'
 '資訊服務類' '電子通路類' '金融業' '認購售權證' '牛熊證(不含展延型牛熊證)' '油電燃氣類' '存託憑證']


In [11]:
# df.to_sql(name='stock_info', con=PG_URI, if_exists='replace', index=False)

### 01-1 Filter out low liquidity stocks
- Focus on twse type stocks
- Focus on normal industry stocks and ETFs
- Remove the stocks with low transaction volumes (Use 2026 Feb monthly transaction volume) 
(https://openapi.twse.com.tw/v1/exchangeReport/FMSRFK_ALL)

In [12]:
target_categories = [
    'ETF', '水泥工業', '食品工業', 
    '電器電纜', '農業科技業', '觀光餐旅', '塑膠工業', '建材營造', 
    '汽車工業', '電子零組件業', '紡織纖維', '貿易百貨', '運動休閒', 
    '電子工業', '電機機械', '生技醫療業', '電腦及週邊設備業', '化學工業', 
    '其他電子業', '玻璃陶瓷', '造紙工業', '鋼鐵工業', '居家生活', 
    '橡膠工業', '航運業', '半導體業', '通信網路業', '光電業', 
    '電子通路業', '資訊服務業', '油電燃氣業', '數位雲端', '金融保險', 
    '文化創意業', '綠能環保', '電子商務業'
]
target_type = ['twse']
filter_df = df[df['industry_category'].isin(target_categories) & df['type'].isin(target_type)]
print(df.shape, filter_df.shape)

(182343, 5) (1969, 5)


In [13]:
# Filter out stocks with low liquidity
def filter_liquidity(trade_vol_share: int = 100*1000, trade_value_ntd: int = 5000000) -> pd.DataFrame:
    """
    This function filters out stocks that do not meet the specified liquidity criteria.
    Stocks must have a daily trading volume greater than `trade_vol_share` shares or a daily
    trading value greater than `trade_value_ntd` NTD.   
    
    The data is fetched from the Taiwan Stock Exchange (TWSE) API(上市個股月成交資訊), which provides the last month's trading volume 
    and value for all listed stocks. The function processes this data to identify stocks that meet the liquidity 
    requirements and returns a DataFrame containing the filtered stocks.
    """
    # Fetch data and preprocess
    try:
        month_stock_vol = pd.read_json('https://openapi.twse.com.tw/v1/exchangeReport/FMSRFK_ALL')
        month_stock_vol.columns = month_stock_vol.columns.str.lower()
        print(f"Original shape: {month_stock_vol.shape}")

        for col in ['tradevolumeb', 'tradevaluea']:
            month_stock_vol[col] = month_stock_vol[col].astype(str).str.replace(',', '').astype(float)
        filter_vol = month_stock_vol['tradevolumeb'] >= trade_vol_share
        filter_value = month_stock_vol['tradevaluea'] >= trade_value_ntd
        filtered_stocks = month_stock_vol[filter_vol & filter_value]
        print(f"Filtered shape: {filtered_stocks.shape}")
        return filtered_stocks
    except Exception as e:
        print(f"Error fetching or processing data: {e}")
        return pd.DataFrame()
    
filtered_stocks = filter_liquidity()
# filtered_stocks.to_sql(name='high_liquidity_stocks', con=PG_URI, if_exists='replace', index=False)
filtered_stocks.head()

Original shape: (30623, 10)
Filtered shape: (4879, 10)


,month,code,name,highestprice,lowestprice,weightedavgpriceab,transaction,tradevaluea,tradevolumeb,turnoverratio
0,11503,0050,元大台灣50,80.80,71.65,75.83,5191615,3.060851e+11,4.036402e+09,21.93
1,11503,0051,元大中型100,114.05,100.35,108.15,17581,2.626280e+08,2.428361e+06,9.91
2,11503,0052,富邦科技,48.25,42.58,45.24,928296,5.934629e+10,1.311641e+09,52.00
3,11503,0053,元大電子,179.40,159.00,168.85,6368,1.123459e+08,6.653320e+05,13.33
4,11503,0055,元大MSCI金融,34.40,31.00,32.67,10187,1.528978e+08,4.678780e+06,5.83


In [14]:
final_df = filter_df[filter_df['stock_id'].isin(filtered_stocks['code'])].reset_index(drop=True)
final_df['date'] = pd.to_datetime(final_df['date']).dt.date
final_df.to_sql(name='targeted_stock_info', con=PG_URI, if_exists='replace', index=False)
final_df.head()

,industry_category,stock_id,stock_name,type,date
0,ETF,0050,元大台灣50,twse,2026-04-21
1,ETF,0051,元大中型100,twse,2026-04-21
2,ETF,0052,富邦科技,twse,2026-04-21
3,ETF,0053,元大電子,twse,2026-04-21
4,ETF,0055,元大MSCI金融,twse,2026-04-21


In [15]:
final_df.value_counts('industry_category').sort_values(ascending=False).reset_index(name='count').head(10)

,industry_category,count
0,電子工業,457
1,ETF,212
2,電子零組件業,104
3,半導體業,95
4,光電業,71
5,電腦及週邊設備業,66
6,生技醫療業,58
7,建材營造,55
8,電機機械,50
9,金融保險,49


### 01-2 Filter transaction value top 5 stocks of each industry

In [16]:
import json

mapping_df = df[['stock_id', 'industry_category']]
filtered_stocks_n = filtered_stocks.merge(mapping_df, left_on='code', right_on='stock_id', how='left')
for col in ['tradevolumeb', 'tradevaluea']:
    filtered_stocks_n[col] = filtered_stocks_n[col].astype(str).str.replace(',', '').astype(float)

# Select top 5 stocks by trading volume for each industry category
top5_stocks = filtered_stocks_n.groupby(
    'industry_category', group_keys=False
    ).apply(lambda x: x.nlargest(5, 'tradevaluea'))

# Display the results (sorted by industry for easier viewing)
top5_stocks[['code', 'name', 'industry_category', 'tradevaluea']].sort_values(['industry_category', 'tradevaluea'], ascending=[True, False])

# Get the top 5 list
# inspect_categories = ['建材營造', '電子零組件業', '半導體業', '通信網路業']
inspect_categories = ['ETF',  '建材營造', '電子零組件業', '半導體業', '通信網路業']
top5_stocks_code = top5_stocks[top5_stocks['industry_category'].isin(inspect_categories)]['code'].tolist()
selected_stocks_dict = {
    category: top5_stocks[top5_stocks['industry_category'] == category]['code'].tolist()
    for category in inspect_categories
}
output_path = Path("/app/data/top5_stocks_by_category.json")
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(selected_stocks_dict, f, ensure_ascii=False, indent=4)
print(top5_stocks_code)
print(selected_stocks_dict)

/tmp/ipykernel_69742/1644037290.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(lambda x: x.nlargest(5, 'tradevaluea'))


['0050', '00919', '00981A', '00631L', '009816', '2330', '2408', '2344', '6770', '2337', '2543', '2515', '2548', '5521', '2542', '2345', '6442', '2485', '2455', '6285', '2313', '2308', '2383', '3037', '2368']
{'ETF': ['0050', '00919', '00981A', '00631L', '009816'], '建材營造': ['2543', '2515', '2548', '5521', '2542'], '電子零組件業': ['2313', '2308', '2383', '3037', '2368'], '半導體業': ['2330', '2408', '2344', '6770', '2337'], '通信網路業': ['2345', '6442', '2485', '2455', '6285']}


## 02 Fetch stock daily

In [3]:
# Create Date list
START_DATE = datetime.strptime("2023-01-01", "%Y-%m-%d")
END_DATE = datetime.strptime("2026-02-28", "%Y-%m-%d")

# 使用 'MS' (Month Start) 頻率來取得每個月的第一天
date_list = pd.date_range(start=START_DATE, end=END_DATE, freq='MS').strftime("%Y%m%d").tolist()
print(date_list)
# Self-made function
# Get_Stock_Daily_Information('2330', '20251201')

['20230101', '20230201', '20230301', '20230401', '20230501', '20230601', '20230701', '20230801', '20230901', '20231001', '20231101', '20231201', '20240101', '20240201', '20240301', '20240401', '20240501', '20240601', '20240701', '20240801', '20240901', '20241001', '20241101', '20241201', '20250101', '20250201', '20250301', '20250401', '20250501', '20250601', '20250701', '20250801', '20250901', '20251001', '20251101', '20251201', '20260101', '20260201']


In [17]:
# Global variables for date range
START_DATE = "20200101"
END_DATE = datetime.now().strftime("%Y%m%d")
# top3_stocks_code = ['009816', '0050', '00919']
# 使用正確的迴圈邏輯：日期 -> 股票代码
for code in tqdm(top5_stocks_code, desc="Processing stocks", total=len(top5_stocks_code)):
    try:
        code_df = pd.DataFrame(get_twse_stock_data(code, START_DATE, END_DATE))
        num_cols = ['capacity', 'turnover', 'high', 'low', 'close', 'change', 'transaction_volume', 'open']
        code_df[num_cols] = code_df[num_cols].apply(pd.to_numeric, errors='coerce')
        code_df['date'] = pd.to_datetime(code_df['date'], unit='s')
        code_df['stock_code_id'] = code_df['stock_code_id'].astype(str)
        time.sleep(5) # Global sleep to be safe
        code_df.to_sql(name=f'daily_info_{code}', con=PG_URI, if_exists='replace', index=False)
        logging.info(f"Finished processing {code}, date range: {START_DATE} to {END_DATE}")
    except Exception as e:
        logging.error(f"Error processing {code}: {e}")

code_df.head()

Processing stocks:   0%|          | 0/25 [00:00<?, ?it/s]

Processing stocks:   0%|          | 0/25 [00:34<?, ?it/s]


KeyboardInterrupt: 

## END